In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# SCD Type 1: Bronze → Silver

Moves the customer dimension from bronze to silver using **SCD Type 1** (overwrite on match, insert if new). A watermark timestamp parameter controls incremental vs. full load:
* **No watermark** → full load (overwrite the entire silver table)
* **Watermark provided** → incremental upsert (only rows with `read_timestamp` greater than the watermark are merged)

In [0]:
# ── Parameter: Watermark Timestamp ──────────────────────────────
# Leave EMPTY for full load. Provide a timestamp (e.g. "2026-09-19 00:00:00")
# to process only rows with read_timestamp > the given value.
dbutils.widgets.text("watermark_timestamp", "", "Watermark Timestamp (empty = full load)")

watermark_ts = dbutils.widgets.get("watermark_timestamp")
is_incremental = bool(watermark_ts and watermark_ts.strip())

print(f"Watermark timestamp : '{watermark_ts}'")
print(f"Load mode          : {'INCREMENTAL (upsert)' if is_incremental else 'FULL LOAD (overwrite)'}")

In [0]:
# ── Read bronze & apply watermark filter ─────────────────────────
bronze_df = spark.table("slowly_changing_dim.bronze.scd1")

if is_incremental:
    bronze_df = bronze_df.filter(
        F.col("read_timestamp") > F.to_timestamp(F.lit(watermark_ts))
    )
    print(f"Filtering rows with read_timestamp > '{watermark_ts}'")
else:
    print("No watermark — processing all rows from bronze")

# Deduplicate on the business key (customer_code), keeping the row with
# the latest read_timestamp so we always upsert the most recent version.
silver_df = (
    bronze_df
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("customer_code")
                .orderBy(F.col("read_timestamp").desc())
    ))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    # Keep only dimension columns; add a load_timestamp for audit
    .select(
        "customer_code", "customer_name", "outlet_type",
        "outlet_segment", "city", "state",
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

row_count = silver_df.count()
print(f"Rows to upsert: {row_count}")
display(silver_df)

In [0]:
# ── SCD Type 1 upsert into silver ────────────────────────────────
target_table = "slowly_changing_dim.silver.scd1"
table_exists = spark.catalog.tableExists(target_table)

if is_incremental and table_exists:
    # Incremental: MERGE (SCD1 upsert)
    silver_df.createOrReplaceTempView("vw_scd1_updates")
    spark.sql(f"""
        MERGE INTO {target_table} AS t
        USING vw_scd1_updates AS s
          ON t.customer_code = s.customer_code
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ SCD1 incremental upsert complete — {row_count} rows merged into {target_table}")

elif is_incremental and not table_exists:
    # Edge case: watermark given but table doesn't exist yet → create from filtered data
    print("⚠ Target table does not exist — creating from the watermark-filtered subset.")
    (silver_df.write
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(target_table))
    print(f"✅ Table created with {row_count} rows.")

else:
    # Full load (or first run): overwrite the entire silver table
    (silver_df.write
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(target_table))
    print(f"✅ SCD1 full load complete — {row_count} rows written to {target_table}")